In [1]:
import yaml

with open("../configs/pretrain_project/silica/baselines/config_mobile_cgcnn.yml") as f:
# with open("../configs/pretrain_project/silica/baselines/config_torchmd.yml") as f:
# with open("../configs/pretrain_project/silica/baselines/config_pbc_processed.yml") as f:
# with open("../configs/pretrain_project/silica/baselines/config_pbc_processed_as.yml") as f:
    config = yaml.safe_load(f)

In [2]:
import random
import numpy as np
import torch

def set_seed(seed):
    # https://pytorch.org/docs/stable/notes/randomness.html
    if seed is None:
        return

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    #torch.autograd.set_detect_anomaly(True)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [3]:
set_seed(config["task"]["seed"])

In [4]:
from matdeeplearn.trainers.base_trainer import BaseTrainer

dataset = BaseTrainer._load_dataset(config["dataset"], config["task"]["run_mode"]) if "src" in config["dataset"] else None
model1 = BaseTrainer._load_model(config["model"], config["dataset"]["preprocess_params"], dataset, 1, 0)[0]
model2 = BaseTrainer._load_model(config["model"], config["dataset"]["preprocess_params"], dataset, 1, 0)[0]
sampler = BaseTrainer._load_sampler(config["optim"], dataset, 1, 0) if "src" in config["dataset"] else None

/net/csefiles/coc-fung-cluster/Qianyu/stable_md/MatDeepLearn_dev/matdeeplearn/preprocessor/datasets.py:25: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.data, self.slic

In [15]:
import torch

checkpoint_pth = "../results/cgcnn/2024-06-14-09-41-25-357-cgcnn_sio2/checkpoint_0/best_checkpoint.pt"
# checkpoint_pth = "../results/2024-09-18-22-36-39-925-silica_torchmd/checkpoint_0/best_checkpoint.pt"
# checkpoint_pth = "../results/2024-10-26-23-56-08-189-graphormer3d_pbc_gbf/checkpoint_0/best_checkpoint.pt"
# checkpoint_pth = "../results/2024-11-03-16-24-35-739-graphormer3d_pbc_gbf_as_no_force_head/checkpoint_0/best_checkpoint.pt"

model1.load_state_dict(torch.load(checkpoint_pth, map_location="cpu")["state_dict"])
model2.load_state_dict(torch.load(checkpoint_pth, map_location="cpu")["state_dict"])

/tmp/ipykernel_4074617/2306521179.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model1.load_state_dict(torch.load(checkpoint_pth, map_location="cpu")["state_dict"])
/t

<All keys matched successfully>

In [5]:
from torch_geometric.loader import DataLoader
from matdeeplearn.preprocessor.pbc_transform import Batch

loader = DataLoader(dataset['test'], batch_size=1, shuffle=False, sampler=sampler)
# loader.collate_fn = Batch.from_datalist

In [17]:
import torch.nn.functional as F
from matdeeplearn.modules.loss import ForceLoss

loss = ForceLoss(weight_energy=0.01, weight_force=50.0)
model1 = model1.to("cuda")
model1.eval()

energy_losses = []
force_losses = []
n_atoms = []
with torch.no_grad():
    for data in loader:
        data = data.to("cuda")
        out = model1(data)
        e_loss = F.l1_loss(out["output"], data.y)
        f_loss = F.l1_loss(out["pos_grad"], data.forces)
        n_atoms.append(data.forces.size(0))
        energy_losses.append(e_loss.item())
        force_losses.append(f_loss.item())

avg_energy_loss = sum(energy_losses) / len(energy_losses)
avg_force_loss = sum(force_losses) / len(force_losses)

print(f"Energy loss: {avg_energy_loss:.5f}, scaled: {0.01 * avg_energy_loss:.5f}")
print(f"Force loss: {avg_force_loss:.5f}, scaled: {50 * avg_force_loss:.5f}")
print(f"Scaled total loss: {0.01 * avg_energy_loss + 50 * avg_force_loss:.5f}")

Energy loss: 26.39121, scaled: 0.26391
Force loss: 0.14325, scaled: 7.16274
Scaled total loss: 7.42665


In [18]:
import numpy as np

force_loss = np.array(force_losses)
n_atoms = np.array(n_atoms)
np.corrcoef(force_loss, n_atoms)

array([[1.        , 0.51148085],
       [0.51148085, 1.        ]])